In [1]:
%%capture
#!pip install -U lightautoml
!pip install flaml[automl] matplotlib openml

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
from flaml import AutoML

from flaml.automl.model import LGBMEstimator

In [5]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
#torch.set_num_threads(N_THREADS)

train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
train = train.drop(columns=['id'])

train_x = train.drop(columns=['efficiency'])
train_y = train['efficiency']

In [8]:
automl = AutoML()
# best: 1000, holdout
settings = {
    "time_budget": 2000,
    "metric": "mae",
    "estimator_list": ["lgbm"],
    "task": "regression",
    "log_file_name": "experiment.log",
    "seed": 41,
    "eval_method": "cv"
}
automl.fit(X_train = train_x, y_train = train_y, **settings)

[flaml.automl.logger: 05-28 09:35:03] {1752} INFO - task = regression
[flaml.automl.logger: 05-28 09:35:03] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 05-28 09:35:03] {1862} INFO - Minimizing error metric: mae
[flaml.automl.logger: 05-28 09:35:03] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 05-28 09:35:03] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 05-28 09:35:03] {2417} INFO - Estimated sufficient time budget=5344s. Estimated necessary time budget=5s.
[flaml.automl.logger: 05-28 09:35:03] {2466} INFO -  at 0.8s,	estimator lgbm's best error=0.0843,	best estimator lgbm's best error=0.0843
[flaml.automl.logger: 05-28 09:35:03] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 05-28 09:35:04] {2466} INFO -  at 1.3s,	estimator lgbm's best error=0.0622,	best estimator lgbm's best error=0.0622
[flaml.automl.logger: 05-28 09:35:04] {2282} INFO - iteration 2, current learner lgbm
[flaml.automl.log

In [9]:
#print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

In [10]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = automl.predict(test)
test_predictions

array([0.40387228, 0.53907557, 0.52104454, ..., 0.60133984, 0.42505131,
       0.5559294 ])

In [11]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.403872
1,1,0.539076
2,2,0.521045
3,3,0.468368
4,4,0.468721
...,...,...
11995,11995,0.547567
11996,11996,0.448981
11997,11997,0.601340
11998,11998,0.425051


In [12]:
submission.to_csv('submission.csv', index = False)